# Session 3: how sure are we about k at a temperature we did not measure?
**A. Real-world problem.** A lab measures k between 600 and 1000 K, but the engine model needs
k at 2000 K. A single fitted line gives a number; a useful answer gives the number *and* a
range that the truth falls inside about 95 times out of 100.

**Plan.** (1) covariance of the fit, (2) the confidence band, (3) Monte Carlo prediction,
(4) validate with synthetic data of known truth, (5) weights, (6) two ways the method fails.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

R_GAS = 8.314462618          # J/(mol K)
A_TRUE = 1.0e13              # 1/s
EA_TRUE = 150.0              # kJ/mol
T_DATA = np.array([600.0, 650.0, 700.0, 750.0, 800.0, 850.0, 900.0, 950.0, 1000.0])   # K

# two-sided 95 percent Student t factors by degrees of freedom
_T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262,
        10: 2.228, 12: 2.179, 15: 2.131, 20: 2.086, 30: 2.042}


def t95(df):
    """Student t factor for a 95 percent two-sided interval (table; next lower tabulated df, so slightly wide)."""
    if df >= 30:
        return 2.042 if df < 120 else 1.98
    return _T95[max(d for d in _T95 if d <= df)]

## Step 1: model, data, and the covariance of the fit
With x = 1000/T and y = ln k the model is y = b0 + b1 x, b0 = ln A, b1 = -Ea/(1000 R).
Least squares gives the covariance of (b0, b1) as s^2 (X^T X)^-1 where X has a column of ones
and a column of x, and s^2 = RSS/(n-2) estimates the error variance of ln k.
Ea = -R b1 (kJ/mol), so the covariance of (ln A, Ea) follows by a sign flip and a scale.

In [ ]:
def arrhenius_k(T, A=A_TRUE, Ea_kJ=EA_TRUE):
    return A * np.exp(-Ea_kJ * 1000.0 / (R_GAS * np.asarray(T, dtype=float)))


def synthetic(T, rel_noise=0.05, seed=0, A=A_TRUE, Ea_kJ=EA_TRUE):
    rng = np.random.default_rng(seed)
    T = np.asarray(T, dtype=float)
    return arrhenius_k(T, A, Ea_kJ) * np.exp(rel_noise * rng.standard_normal(T.size))


def fit_with_cov(T, k, sigma_ln=None):
    """Fit ln k = b0 + b1 * 1000/T. sigma_ln: known error of ln k per point (weighted fit) or None (estimated).
    Returns dict: b (b0, b1), cov (2x2 of b), df, s (error of ln k used)."""
    T, k = np.asarray(T, dtype=float), np.asarray(k, dtype=float)
    if T.size < 3:
        raise ValueError("need at least 3 points to estimate the error of a straight-line fit")
    x, y = 1000.0 / T, np.log(k)
    X = np.column_stack([np.ones(x.size), x])
    if sigma_ln is None:
        b = np.linalg.solve(X.T @ X, X.T @ y)
        df = x.size - 2
        s = float(np.sqrt(np.sum((y - X @ b) ** 2) / df))
        cov = s ** 2 * np.linalg.inv(X.T @ X)
    else:
        w = 1.0 / np.asarray(sigma_ln, dtype=float) ** 2
        XtW = X.T * w
        b = np.linalg.solve(XtW @ X, XtW @ y)
        df = None
        s = 1.0
        cov = np.linalg.inv(XtW @ X)
    return {"b": b, "cov": cov, "df": df, "s": s}


def lnA_Ea(fit):
    """Parameters (ln A, Ea in kJ/mol), their standard errors and their correlation."""
    M = np.diag([1.0, -R_GAS])
    cov = M @ fit["cov"] @ M.T
    se = np.sqrt(np.diag(cov))
    return {"lnA": float(fit["b"][0]), "Ea": float(-R_GAS * fit["b"][1]), "se_lnA": float(se[0]),
            "se_Ea": float(se[1]), "corr": float(cov[0, 1] / (se[0] * se[1])), "cov": cov}

## Step 2: the confidence band for ln k(T)
At temperature T the fitted ln k is g^T b with g = (1, 1000/T). Its variance is g^T cov g,
which equals s^2 [1/n + (x - xbar)^2 / Sxx]. It is smallest at the mean x and grows with
distance from it: the band flares outside the data. A 95 percent interval is
ln k +- t * se; on the k axis the band is a *factor* exp(+-t se).

In [ ]:
def se_ln_k(fit, T):
    g = np.column_stack([np.ones(np.size(T)), 1000.0 / np.asarray(T, dtype=float)])
    return np.sqrt(np.einsum("ij,jk,ik->i", g, fit["cov"], g))


def band(fit, T):
    """(central k, lower k, upper k) with a 95 percent interval for the mean ln k."""
    T = np.atleast_1d(np.asarray(T, dtype=float))
    mid = fit["b"][0] + fit["b"][1] * 1000.0 / T
    t = 1.96 if fit["df"] is None else t95(fit["df"])
    h = t * se_ln_k(fit, T)
    return np.exp(mid), np.exp(mid - h), np.exp(mid + h)


def expected_factor(sigma_ln, n, T_low, T_high=1000.0, T_target=2000.0):
    """Design tool: half-width factor exp(t * se) of the 95 percent interval at T_target for n points
    evenly spaced in 1000/T between T_low and T_high, if the error of ln k is sigma_ln (no data needed)."""
    x = np.linspace(1000.0 / T_high, 1000.0 / T_low, n)
    sxx = float(np.sum((x - x.mean()) ** 2))
    se = sigma_ln * np.sqrt(1.0 / n + (1000.0 / T_target - x.mean()) ** 2 / sxx)
    return float(np.exp(t95(n - 2) * se))

## Step 3: Monte Carlo propagation
Draw many parameter pairs from a normal distribution with the fitted covariance, compute k at
the target temperature for each, and take the 2.5 and 97.5 percentiles. It agrees with the
formula above and keeps working for functions of the parameters that have no simple formula.

In [ ]:
def mc_predict(fit, T_target, draws=40000, seed=7):
    """Returns (median k, 2.5 percent, 97.5 percent) at T_target from parameter draws."""
    rng = np.random.default_rng(seed)
    b = rng.multivariate_normal(fit["b"], fit["cov"], size=draws)
    k = np.exp(b[:, 0] + b[:, 1] * 1000.0 / T_target)
    lo, med, hi = np.percentile(k, [2.5, 50, 97.5])
    return float(med), float(lo), float(hi)

## Step 4: validate against known truth
Because we made the data from known A and Ea, we can repeat the whole experiment many times and
count how often the 95 percent interval at the target temperature contains the true k.
A correct method gives a coverage close to 0.95.

In [ ]:
def coverage(T, T_target, rel_noise=0.05, repeats=2000, seed=11, A=A_TRUE, Ea_kJ=EA_TRUE):
    """Fraction of repeats whose 95 percent interval at T_target contains the true k, and the mean fitted Ea."""
    rng = np.random.default_rng(seed)
    T = np.asarray(T, dtype=float)
    true = float(arrhenius_k(T_target, A, Ea_kJ))
    hit, eas = 0, []
    for _ in range(repeats):
        k = arrhenius_k(T, A, Ea_kJ) * np.exp(rel_noise * rng.standard_normal(T.size))
        f = fit_with_cov(T, k)
        _, lo, hi = band(f, T_target)
        hit += int(lo[0] <= true <= hi[0])
        eas.append(lnA_Ea(f)["Ea"])
    return hit / repeats, float(np.mean(eas))

## Step 5: weights
If every point has the same relative error, ln k has the same error everywhere and plain
least squares is already right. If the lab is much less sure about some points (say 30 percent at
the lowest temperatures, 3 percent at the highest), weight each point by 1/sigma^2.

In [ ]:
def uneven_experiment(T, sig_lo=0.30, sig_hi=0.03, seed=0):
    """Relative error falls linearly with T from sig_lo (lowest T) to sig_hi (highest T)."""
    T = np.asarray(T, dtype=float)
    sig = sig_lo + (sig_hi - sig_lo) * (T - T.min()) / (T.max() - T.min())
    rng = np.random.default_rng(seed)
    return arrhenius_k(T) * np.exp(sig * rng.standard_normal(T.size)), sig


def ea_scatter_weighted_vs_plain(T, repeats=1500, seed=5):
    """Standard deviation of fitted Ea, plain versus weighted, over repeated uneven experiments."""
    plain, weighted = [], []
    for i in range(repeats):
        k, sig = uneven_experiment(T, seed=seed + i)
        plain.append(lnA_Ea(fit_with_cov(T, k))["Ea"])
        weighted.append(lnA_Ea(fit_with_cov(T, k, sigma_ln=sig))["Ea"])
    return float(np.std(plain, ddof=1)), float(np.std(weighted, ddof=1))

## Step 6: two failures
(a) A narrow 1000/T range makes Ea (and the extrapolation) very uncertain.
(b) The straight line assumes ONE Arrhenius process. Two parallel channels with different
Ea give a curved plot; the line is then wrong and the band, which only counts noise, does
not know it.

In [ ]:
def two_channel_k(T, A1=1.0e7, E1=50.0, A2=1.0e14, E2=200.0):
    """Sum of two Arrhenius channels (1/s); the high-Ea channel takes over at high T."""
    return arrhenius_k(T, A1, E1) + arrhenius_k(T, A2, E2)


def straight_line_on_curved(T, rel_noise=0.03, seed=2):
    k = two_channel_k(T) * np.exp(rel_noise * np.random.default_rng(seed).standard_normal(np.size(T)))
    return fit_with_cov(T, k)

## Step 7: see it
Left: the data, the fitted line, the 95 percent band out to 2000 K and the true law.
Middle: the one-sigma ellipse of (ln A, Ea), long and thin because the two are correlated.
Right: the straight-line fit on a two-channel reaction misses the truth at 2000 K.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    T = T_DATA
    k = synthetic(T, 0.05, seed=4)
    f = fit_with_cov(T, k)
    q = lnA_Ea(f)
    print("Ea", round(q["Ea"], 2), "+-", round(q["se_Ea"], 2), "lnA", round(q["lnA"], 3), "+-", round(q["se_lnA"], 3),
          "corr", round(q["corr"], 4))
    mid, lo, hi = band(f, 2000.0)
    print("k(2000 K)", mid[0], lo[0], hi[0], "true", float(arrhenius_k(2000.0)))
    print("MC", mc_predict(f, 2000.0))

    fig, ax = plt.subplots(1, 3, figsize=(13, 3.9))
    Tg = 1000.0 / np.linspace(0.45, 1.8, 120)
    m, l, h = band(f, Tg)
    ax[0].fill_between(1000 / Tg, np.log(l), np.log(h), alpha=0.3, label="95 percent band")
    ax[0].plot(1000 / Tg, np.log(m), label="fit")
    ax[0].plot(1000 / Tg, np.log(arrhenius_k(Tg)), "k:", label="true law")
    ax[0].plot(1000 / T, np.log(k), "o", color="tab:red", label="data")
    ax[0].set_xlabel("1000 / T (1/K x 1000)")
    ax[0].set_ylabel("ln k (k in 1/s)")
    ax[0].set_title("The band flares outside the data")
    ax[0].legend(fontsize=8)
    ax[0].grid(alpha=0.3)

    th = np.linspace(0, 2 * np.pi, 100)
    w, V = np.linalg.eigh(q["cov"])
    ell = V @ (np.sqrt(w)[:, None] * np.array([np.cos(th), np.sin(th)]))
    ax[1].plot(q["Ea"] + ell[1], q["lnA"] + ell[0])
    ax[1].plot(q["Ea"], q["lnA"], "o")
    ax[1].plot(EA_TRUE, np.log(A_TRUE), "k*", ms=10, label="truth")
    ax[1].set_xlabel("Ea (kJ/mol)")
    ax[1].set_ylabel("ln A (A in 1/s)")
    ax[1].set_title(f"One-sigma ellipse, correlation {q['corr']:.3f}")
    ax[1].legend(fontsize=8)
    ax[1].grid(alpha=0.3)

    Tc = np.linspace(800, 1600, 9)
    fc = straight_line_on_curved(Tc)
    Tg2 = 1000.0 / np.linspace(0.45, 1.3, 80)
    mc, lc, hc = band(fc, Tg2)
    ax[2].plot(1000 / Tg2, np.log(two_channel_k(Tg2)), "k:", label="true (two channels)")
    ax[2].fill_between(1000 / Tg2, np.log(lc), np.log(hc), alpha=0.3, label="95 percent band")
    ax[2].plot(1000 / Tg2, np.log(mc), label="straight-line fit")
    ax[2].plot(1000 / Tc, np.log(two_channel_k(Tc)), "o", color="tab:red", label="data")
    ax[2].set_xlabel("1000 / T (1/K x 1000)")
    ax[2].set_ylabel("ln k (k in 1/s)")
    ax[2].set_title("Failure: a curved Arrhenius plot")
    ax[2].legend(fontsize=8)
    ax[2].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "band_ellipse_failure.png", dpi=120)
    print("coverage at 2000 K:", coverage(T, 2000.0, repeats=600))
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print the correlation of (ln A, Ea) for the windows 600 to 1000 K and 900 to 1000 K with `lnA_Ea` and explain the change.
2. **Stretch.** Run `coverage(T, 2000.0, rel_noise=0.15)` and say why the coverage hardly changes although the interval is three times wider.
3. **Explain.** Explain, in your own words and with the formula, why the middle of the data is where we know k best, and why that says nothing about the model being right.

## Assessment
- **A. Conceptual.** Why does the confidence band get wider outside the data, and why is it widest on the side of the extrapolation?
- **B. Numerical.** With s = 0.0535, n = 9, xbar = 1.2841 and Sxx = 0.4113 find the standard error of ln k at 800 K (x = 1.25) by hand and the 95 percent factor with t = 2.365.
- **C. Graph and simulation interpretation.** In the right panel of the figure, the straight-line fit has a band that does not contain the truth at 2000 K. What number from the fit warns you that the straight line is wrong?
- **D. Design and optimisation challenge.** Design: you can add measurements to the 600 to 1000 K data to get k(2000 K) with a factor of at most 1.15. What do you change first, the number of points or the range, and why? Test the cases in the app.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    T = L.T_DATA
    x = 1000.0 / T
    xb = x.mean()
    sxx = float(np.sum((x - xb) ** 2))
    k = L.synthetic(T, 0.05, 4)
    f = L.fit_with_cov(T, k)
    q = L.lnA_Ea(f)
    checks.append(("worked example: Sxx = 0.4113 and mean of 1000/T = 1.2841 (T = 778.7 K)", abs(sxx - 0.4113) < 1e-3 and abs(xb - 1.2841) < 1e-4))
    checks.append(("worked example: s = 0.0535 on 7 degrees of freedom", abs(f["s"] - 0.0535) < 1e-4 and f["df"] == 7))
    checks.append(("worked example: Ea = 149.43 +- 0.69 kJ/mol, ln A = 29.833 +- 0.109", abs(q["Ea"] - 149.43) < 0.01 and abs(q["se_Ea"] - 0.69) < 0.01 and abs(q["lnA"] - 29.833) < 0.001 and abs(q["se_lnA"] - 0.109) < 0.001))
    checks.append(("worked example: ln A and Ea are correlated, r = 0.986", abs(q["corr"] - 0.986) < 0.001))
    se = f["s"] * np.sqrt(1 / 9 + (0.5 - xb) ** 2 / sxx)
    checks.append(("worked example: se of ln k at 2000 K by hand = 0.0678", abs(se - 0.0678) < 1e-4 and abs(float(L.se_ln_k(f, [2000.0])[0]) - se) < 1e-12))
    mid, lo, hi = L.band(f, 2000.0)
    checks.append(("worked example: k(2000 K) = 1.13e9 per s, 95 percent interval 9.6e8 to 1.33e9", abs(mid[0] / 1.1315e9 - 1) < 0.001 and abs(lo[0] / 9.64e8 - 1) < 0.002 and abs(hi[0] / 1.328e9 - 1) < 0.002))
    checks.append(("the true k(2000 K) = 1.209e9 lies inside that interval", lo[0] < float(L.arrhenius_k(2000.0)) < hi[0] and abs(float(L.arrhenius_k(2000.0)) / 1.2091e9 - 1) < 1e-3))
    checks.append(("the band is narrowest at the mean of 1000/T: se = s/3 = 0.0178", abs(float(L.se_ln_k(f, [1000.0 / xb])[0]) - f["s"] / 3) < 1e-12))
    ratios = [float(L.band(f, t)[2][0] / L.band(f, t)[1][0]) for t in (800.0, 1500.0, 2000.0)]
    checks.append(("band width (upper/lower) flares: 1.09 at 800 K, 1.29 at 1500 K, 1.38 at 2000 K", abs(ratios[0] - 1.089) < 0.002 and abs(ratios[1] - 1.294) < 0.002 and abs(ratios[2] - 1.378) < 0.002))
    med, mlo, mhi = L.mc_predict(f, 2000.0)
    z = 1.96 * float(L.se_ln_k(f, [2000.0])[0])
    checks.append(("Monte Carlo median and the 1.96-sigma formula agree within 0.5 percent", abs(med / mid[0] - 1) < 0.005 and abs(mlo / (mid[0] * np.exp(-z)) - 1) < 0.015 and abs(mhi / (mid[0] * np.exp(z)) - 1) < 0.015))
    checks.append(("t factor for 3 degrees of freedom is 3.182 and for 7 is 2.365", abs(L.t95(3) - 3.182) < 1e-9 and abs(L.t95(7) - 2.365) < 1e-9))
    cov_p, ea_mean = L.coverage(T, 2000.0, repeats=2000)
    checks.append(("validation: 95 percent interval at 2000 K covers the truth 94.8 percent of 2000 repeats", abs(cov_p - 0.9475) < 0.001 and 0.93 < cov_p < 0.97))
    checks.append(("validation: fitted Ea averages 150.00 kJ/mol over repeats (unbiased)", abs(ea_mean - 150.0) < 0.01))
    cov_in, _ = L.coverage(T, 800.0, repeats=2000)
    checks.append(("validation: coverage inside the data (800 K) is also close to 95 percent", 0.93 < cov_in < 0.97))
    # narrow range failure
    Tn = np.linspace(900.0, 1000.0, 9)
    fn = L.fit_with_cov(Tn, L.synthetic(Tn, 0.05, 4))
    qn = L.lnA_Ea(fn)
    mn, ln_, hn = L.band(fn, 2000.0)
    checks.append(("failure: 900-1000 K only gives se(Ea) = 4.07 kJ/mol (5.9 times the 0.69 of the wide range)", abs(qn["se_Ea"] - 4.07) < 0.01 and abs(qn["se_Ea"] / q["se_Ea"] - 5.87) < 0.05))
    checks.append(("failure: the 2000 K interval spans a factor 3.6 (against 1.38)", abs(hn[0] / ln_[0] - 3.62) < 0.02 and qn["corr"] > 0.999))
    covn, _ = L.coverage(Tn, 2000.0, repeats=2000)
    checks.append(("narrow data are wide but honest: coverage is still about 95 percent (94.2)", abs(covn - 0.9415) < 0.001))
    # weights
    ps, ws = L.ea_scatter_weighted_vs_plain(np.linspace(600.0, 1000.0, 9))
    checks.append(("weights: uneven errors give Ea scatter 2.92 plain and 1.97 weighted kJ/mol", abs(ps - 2.92) < 0.03 and abs(ws - 1.97) < 0.03 and ws < ps))
    fw = L.fit_with_cov(T, k, sigma_ln=np.full(9, 0.05))
    checks.append(("weights: equal sigma gives the same Ea and se(Ea) = R sigma / sqrt(Sxx) = 0.648", abs(L.lnA_Ea(fw)["Ea"] - q["Ea"]) < 1e-9 and abs(L.lnA_Ea(fw)["se_Ea"] - 0.648) < 0.001))
    # curved plot
    Tc = np.linspace(800.0, 1600.0, 9)
    fc = L.straight_line_on_curved(Tc)
    mc, lc, hc = L.band(fc, 2000.0)
    true2 = float(L.two_channel_k(2000.0))
    checks.append(("failure: two-channel truth at 2000 K is 5.98e8 per s, the line predicts 6.5e7 (9.2 times too low)", abs(true2 / 5.984e8 - 1) < 1e-3 and abs(true2 / mc[0] - 9.23) < 0.05))
    checks.append(("failure: the true value is outside the 95 percent band", not (lc[0] <= true2 <= hc[0])))
    fs = L.fit_with_cov(Tc, L.synthetic(Tc, 0.03, 2))
    checks.append(("warning sign: the residual error s is 0.86 for the curved data, 0.037 for a true single channel", abs(fc["s"] - 0.856) < 0.005 and abs(fs["s"] - 0.0374) < 0.001))
    try:
        L.fit_with_cov([600.0, 1000.0], L.arrhenius_k([600.0, 1000.0]))
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge case: 2 points leave no degrees of freedom, so the fit refuses to give an error", raised))
    three = L.fit_with_cov([600.0, 800.0, 1000.0], L.synthetic([600.0, 800.0, 1000.0], 0.05, 1))
    checks.append(("edge case: 3 points give 1 degree of freedom and t = 12.706", three["df"] == 1 and abs(L.t95(1) - 12.706) < 1e-9))
    Tp = np.linspace(700.0, 1100.0, 7)
    fp = L.fit_with_cov(Tp, L.synthetic(Tp, 0.06, 21, 3e11, 120.0))
    qp = L.lnA_Ea(fp)
    mp, lp, hp = L.band(fp, 2000.0)
    tp = float(L.arrhenius_k(2000.0, 3e11, 120.0))
    checks.append(("mini-project data: Ea = 118.75 +- 1.42 kJ/mol", abs(qp["Ea"] - 118.75) < 0.01 and abs(qp["se_Ea"] - 1.42) < 0.01))
    checks.append(("mini-project: k(2000 K) = 2.0e8 per s with interval 1.5e8 to 2.67e8, true 2.2e8 inside", abs(mp[0] / 2.0e8 - 1) < 0.01 and abs(lp[0] / 1.5e8 - 1) < 0.01 and abs(hp[0] / 2.67e8 - 1) < 0.01 and lp[0] < tp < hp[0] and abs(tp / 2.2e8 - 1) < 0.01))
    checks.append(("mini-project: interval factor upper/lower is 1.78 for 700-1100 K", abs(hp[0] / lp[0] - 1.78) < 0.01))
    Tq = np.linspace(900.0, 1100.0, 7)
    fq = L.fit_with_cov(Tq, L.synthetic(Tq, 0.06, 21, 3e11, 120.0))
    mq, lq, hq = L.band(fq, 2000.0)
    checks.append(("mini-project lever: window 900-1100 K gives factor 3.12, above the limit 2", abs(hq[0] / lq[0] - 3.12) < 0.01))
    ends = np.array([700.0, 700.0, 700.0, 700.0, 1100.0, 1100.0, 1100.0])
    se_of = lambda Tt: 0.06 * np.sqrt(np.array([1.0, 0.5]) @ np.linalg.inv(np.column_stack([np.ones(7), 1000.0 / Tt]).T @ np.column_stack([np.ones(7), 1000.0 / Tt])) @ np.array([1.0, 0.5]))
    checks.append(("design for prediction: expected se of ln k(2000 K) with sigma 0.06 is 0.0866 for even spacing, 0.0663 with points at the ends",
                   abs(se_of(Tp) - 0.0866) < 0.0002 and abs(se_of(ends) - 0.0663) < 0.0002))
    checks.append(("design: 6 points from 800 K, 5 percent error: factor 1.52 on k(2000 K); 12 points from 500 K: 1.11",
                   abs(L.expected_factor(0.05, 6, 800.0) - 1.52) < 0.005 and abs(L.expected_factor(0.05, 12, 500.0) - 1.113) < 0.002))
    checks.append(("app challenge: 12 points from 500 K meets the 1.15 limit, the default (6 points from 800 K) does not",
                   L.expected_factor(0.05, 12, 500.0) <= 1.15 < L.expected_factor(0.05, 6, 800.0)))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
